# SENTIMENT DEVELOPMENT EXPERIMENT 02

This notebook trains the pinned SinBERT-small checkpoint on the frozen Dev-v2 Sinhala train split and selects by validation macro-F1. It never opens the frozen-test CSV; only the frozen manifest IDs are read for contamination checks. This is development/demo evidence, not a final benchmark.

In [1]:
import csv, hashlib, json, random, re, sys, time
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support, f1_score
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from transformers.models.roberta.modeling_roberta import RobertaClassificationHead

REPO = Path(r'C:\Users\Yasindu\Desktop\Chat_Research\IT22638168')
SENTIMENT = REPO / 'ml' / 'sentiment'
DATASET = SENTIMENT / 'outputs' / 'development_v2' / 'release' / 'maternalink_sinhala_dev_v2.csv'
MANIFEST = SENTIMENT / 'outputs' / 'development_v2' / 'release' / 'maternalink_sinhala_dev_v2.manifest.json'
MEMBERSHIP = SENTIMENT / 'outputs' / 'development_v2' / 'release' / 'maternalink_sinhala_dev_v2_split_membership.csv'
FROZEN_MANIFEST = SENTIMENT / 'data' / 'processed' / 'FROZEN_TEST_SET_MANIFEST.md'
OUT = SENTIMENT / 'outputs' / 'development_v2' / 'experiment_02'
OUT.mkdir(parents=True, exist_ok=True)
MODEL_REV = '7059f20a28a2b1e2ff2f45b13d6956435cdacb6a'
MODEL_PATH = Path(r'C:\Users\Yasindu\.cache\huggingface\hub\models--sinhala-nlp--sinhala-sentiment-analysis-sinbert-small\snapshots\7059f20a28a2b1e2ff2f45b13d6956435cdacb6a\best_model')
SEED = 20260828
LABELS = {'CALM': 0, 'NEUTRAL': 1, 'DISTRESSED': 2}
ORDER = ['CALM', 'NEUTRAL', 'DISTRESSED']
MAX_LENGTH, BATCH_SIZE, EPOCHS = 512, 8, 5
assert DATASET.exists() and MANIFEST.exists() and MEMBERSHIP.exists()
assert MODEL_PATH.exists() and MODEL_REV in str(MODEL_PATH)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()

def frozen_ids(path):
    return {m.group(1) for m in re.finditer(r'(?m)^- ([A-Z0-9-]+)$', path.read_text(encoding='utf-8'))}

manifest = json.loads(MANIFEST.read_text(encoding='utf-8'))
assert sha256(DATASET) == manifest['dataset_sha256']
assert sha256(MEMBERSHIP) == manifest['split']['membership_sha256']
frozen = frozen_ids(FROZEN_MANIFEST)
print('Verified Dev-v2 and split hashes; frozen manifest IDs loaded only:', len(frozen))

C:\Users\Yasindu\.conda\envs\sentiment-model\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Verified Dev-v2 and split hashes; frozen manifest IDs loaded only: 120


In [2]:
development = pd.read_csv(DATASET)
membership = pd.read_csv(MEMBERSHIP)
assert set(development.language) == {'si'}
assert set(development.adjudicated_label) == set(LABELS)
assert development.record_id.is_unique
assert set(development.record_id) == set(membership.record_id)
assert not set(development.record_id) & frozen
assert not set(membership[membership.split == 'train'].record_id) & set(membership[membership.split == 'validation'].record_id)
assert membership.split.value_counts().to_dict() == {'train': 223, 'validation': 76}
assert sha256(MEMBERSHIP) == manifest['split']['membership_sha256']
train_ids = membership.loc[membership.split == 'train', 'record_id'].tolist()
val_ids = membership.loc[membership.split == 'validation', 'record_id'].tolist()
train = development[development.record_id.isin(train_ids)].copy()
val = development[development.record_id.isin(val_ids)].copy()
assert len(train) == 223 and len(val) == 76
assert train.adjudicated_label.value_counts().reindex(ORDER).tolist() == [52, 137, 34]
assert val.adjudicated_label.value_counts().reindex(ORDER).tolist() == [18, 46, 12]
print('Train counts:', train.adjudicated_label.value_counts().reindex(ORDER).to_dict())
print('Validation counts:', val.adjudicated_label.value_counts().reindex(ORDER).to_dict())

Train counts: {'CALM': 52, 'NEUTRAL': 137, 'DISTRESSED': 34}
Validation counts: {'CALM': 18, 'NEUTRAL': 46, 'DISTRESSED': 12}


In [3]:
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_PATH), local_files_only=True)
model.classifier = RobertaClassificationHead(model.config)
model.config.num_labels = 3
model.config.id2label = {str(v): k for k, v in LABELS.items()}
model.config.label2id = LABELS
model.config.problem_type = 'single_label_classification'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

class MoodDataset(Dataset):
    def __init__(self, frame): self.frame = frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        return row.text, LABELS[row.adjudicated_label]

def collate(batch):
    texts, labels = zip(*batch)
    enc = tokenizer(list(texts), truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors='pt')
    enc['labels'] = torch.tensor(labels, dtype=torch.long)
    return enc

train_loader = DataLoader(MoodDataset(train), batch_size=BATCH_SIZE, shuffle=True, generator=torch.Generator().manual_seed(SEED), collate_fn=collate)
val_loader = DataLoader(MoodDataset(val), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
train_counts = np.bincount([LABELS[x] for x in train.adjudicated_label], minlength=3)
class_weights = torch.tensor(len(train) / (3 * train_counts), dtype=torch.float32, device=device)
expected_weights = np.array([223/(3*52), 223/(3*137), 223/(3*34)], dtype=np.float32)
assert np.allclose(class_weights.detach().cpu().numpy(), expected_weights)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=max(1, round(EPOCHS * len(train_loader) * 0.1)), num_training_steps=EPOCHS * len(train_loader))
print('Checkpoint:', MODEL_REV, '| device:', device)
print('Class weights [CALM, NEUTRAL, DISTRESSED]:', class_weights.detach().cpu().tolist())

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5411.01it/s]

Checkpoint: 7059f20a28a2b1e2ff2f45b13d6956435cdacb6a | device: cpu
Class weights [CALM, NEUTRAL, DISTRESSED]: [1.4294872283935547, 0.5425790548324585, 2.186274528503418]


In [4]:
def evaluate():
    model.eval(); losses=[]; yt=[]; yp=[]; probs=[]
    with torch.no_grad():
        for batch in val_loader:
            labels = batch.pop('labels').to(device); inputs = {k:v.to(device) for k,v in batch.items()}
            logits = model(**inputs).logits; losses.append(float(criterion(logits, labels).item()))
            yt.extend(labels.cpu().tolist()); yp.extend(logits.argmax(1).cpu().tolist()); probs.extend(torch.softmax(logits, dim=-1).cpu().numpy())
    p, r, f, s = precision_recall_fscore_support(yt, yp, labels=[0,1,2], zero_division=0)
    return {'loss': float(np.mean(losses)), 'accuracy': float(accuracy_score(yt, yp)), 'macro_f1': float(f1_score(yt, yp, labels=[0,1,2], average='macro', zero_division=0)), 'weighted_f1': float(f1_score(yt, yp, labels=[0,1,2], average='weighted', zero_division=0)), 'per_class': {ORDER[i]: {'precision': float(p[i]), 'recall': float(r[i]), 'f1': float(f[i]), 'support': int(s[i])} for i in range(3)}, 'confusion_matrix': confusion_matrix(yt, yp, labels=[0,1,2]).tolist(), 'y_true': yt, 'y_pred': yp, 'probs': probs}

history=[]; best=None; best_epoch=None
for epoch in range(1, EPOCHS + 1):
    model.train(); train_losses=[]
    for batch in train_loader:
        labels=batch.pop('labels').to(device); inputs={k:v.to(device) for k,v in batch.items()}
        optimizer.zero_grad(set_to_none=True); loss=criterion(model(**inputs).logits, labels); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step(); scheduler.step(); train_losses.append(float(loss.item()))
    metrics=evaluate(); metrics.update({'epoch':epoch, 'train_loss':float(np.mean(train_losses))}); history.append(metrics)
    print({k:v for k,v in metrics.items() if k not in {'per_class','confusion_matrix','y_true','y_pred','probs'}})
    if best is None or metrics['macro_f1'] > best['macro_f1'] or (metrics['macro_f1'] == best['macro_f1'] and metrics['loss'] < best['loss']):
        best=metrics; best_epoch=epoch; model.save_pretrained(OUT/'best_checkpoint'); tokenizer.save_pretrained(OUT/'best_checkpoint')

assert best is not None
print('Best epoch:', best_epoch, 'best validation macro-F1:', best['macro_f1'])

{'loss': 0.9765892267227173, 'accuracy': 0.7105263157894737, 'macro_f1': 0.604884004884005, 'weighted_f1': 0.6869288606130712, 'epoch': 1, 'train_loss': 1.0852547926562173}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.68it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.68it/s]

{'loss': 0.8676589727401733, 'accuracy': 0.5921052631578947, 'macro_f1': 0.552485805290126, 'weighted_f1': 0.6030164214024885, 'epoch': 2, 'train_loss': 0.8979562478406089}


...

{'loss': 0.8196377038955689, 'accuracy': 0.6842105263157895, 'macro_f1': 0.6248506571087216, 'weighted_f1': 0.6846821354461422, 'epoch': 5, 'train_loss': 0.4828503940786634}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.44it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.41it/s]

Best epoch: 5 best validation macro-F1: 0.6248506571087216


In [5]:
pd.DataFrame([{k: v for k, v in row.items() if k not in {'per_class','confusion_matrix','y_true','y_pred','probs'}} for row in history]).to_csv(OUT/'training_history.csv', index=False)
pred_rows=[]
for rid, true, pred, prob in zip(val.record_id.tolist(), best['y_true'], best['y_pred'], best['probs']):
    pred_rows.append({'record_id':rid, 'true_label':ORDER[true], 'predicted_label':ORDER[pred], 'prob_calm':float(prob[0]), 'prob_neutral':float(prob[1]), 'prob_distressed':float(prob[2]), 'max_probability':float(max(prob))})
pd.DataFrame(pred_rows).to_csv(OUT/'validation_predictions.csv', index=False)
cm = pd.DataFrame(best['confusion_matrix'], index=ORDER, columns=ORDER); cm.to_csv(OUT/'confusion_matrix.csv')
metrics = {k:v for k,v in best.items() if k not in {'y_true','y_pred','probs'}}; (OUT/'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
metadata = {'status':'EXPERIMENT_02_DEVELOPMENT_ONLY','dataset_path':str(DATASET),'dataset_sha256':sha256(DATASET),'dataset_version':manifest['dataset_version'],'split_path':str(MEMBERSHIP),'split_sha256':sha256(MEMBERSHIP),'seed':SEED,'model_id':'sinhala-nlp/sinhala-sentiment-analysis-sinbert-small','model_revision':MODEL_REV,'tokenizer_class':type(tokenizer).__name__,'classification_head':'new RobertaClassificationHead with 3 outputs','label_order':ORDER,'device':str(device),'max_length':MAX_LENGTH,'batch_size':BATCH_SIZE,'epochs_requested':EPOCHS,'epochs_completed':len(history),'optimizer':'AdamW','learning_rate':2e-5,'weight_decay':0.01,'warmup_ratio':0.1,'gradient_clipping':1.0,'loss':'CrossEntropyLoss with training-only class weights','class_weights_training_only':class_weights.detach().cpu().tolist(),'train_counts':dict(zip(ORDER, train_counts.tolist())),'train_count':len(train),'validation_count':len(val),'primary_metric':'validation_macro_f1','best_epoch':best_epoch,'frozen_test_used':False,'alternative_models_used':False,'fer_fusion_used':False}
(OUT/'experiment_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
findings = f'''# SENTIMENT DEVELOPMENT EXPERIMENT 02 — Findings

Status: completed once; development evidence only.

Dataset: `{manifest['dataset_version']}` with SHA-256 `{sha256(DATASET)}`. Train/validation: {len(train)}/{len(val)}; split SHA-256 `{sha256(MEMBERSHIP)}`.

Best epoch: {best_epoch}. Validation accuracy: {best['accuracy']:.6f}. Macro-F1: {best['macro_f1']:.6f}. Weighted-F1: {best['weighted_f1']:.6f}.

Per-class metrics:

''' + '\n'.join(f"- {label}: precision={best['per_class'][label]['precision']:.6f}, recall={best['per_class'][label]['recall']:.6f}, F1={best['per_class'][label]['f1']:.6f}, support={best['per_class'][label]['support']}" for label in ORDER) + f'''

Confusion matrix rows=true, columns=predicted: `{best['confusion_matrix']}`.

The model used the pinned SinBERT-small encoder with a new three-class head and training-only weighted cross-entropy. No frozen-test text, labels, predictions, or metrics were accessed; only frozen manifest IDs were used for contamination checks. No alternative model or FER fusion was used.
'''
(OUT/'EXPERIMENT_02_FINDINGS.md').write_text(findings, encoding='utf-8')
print('Saved Experiment 02 artifacts to', OUT)
print(json.dumps({'accuracy':best['accuracy'], 'macro_f1':best['macro_f1'], 'weighted_f1':best['weighted_f1'], 'per_class':best['per_class'], 'confusion_matrix':best['confusion_matrix'], 'best_epoch':best_epoch}, indent=2))

Saved Experiment 02 artifacts to C:\Users\Yasindu\Desktop\Chat_Research\IT22638168\ml\sentiment\outputs\development_v2\experiment_02
{
  "accuracy": 0.6842105263157895,
  "macro_f1": 0.6248506571087216,
  "weighted_f1": 0.6846821354461422,
  "per_class": {
    "CALM": {
      "precision": 0.6153846153846154,
      "recall": 0.4444444444444444,
      "f1": 0.5161290322580645,
      "support": 18
    },
    "NEUTRAL": {
      "precision": 0.7954545454545454,
      "recall": 0.7608695652173914,
      "f1": 0.7777777777777778,
      "support": 46
    },
    "DISTRESSED": {
      "precision": 0.47368421052631576,
      "recall": 0.75,
      "f1": 0.5806451612903226,
      "support": 12
    }
  },
  "confusion_matrix": [
    [
      8,
      8,
      2
    ],
    [
      3,
      35,
      8
    ],
    [
      2,
      1,
      9
    ]
  ],
  "best_epoch": 5
}
